# 🧠 Notebook 18: Scaling Laws & Compute-Optimal Training

How to predict model performance before training, allocate compute optimally, and navigate the emergent abilities debate.

**Prerequisites:** Notebook 08 (Training on Apple Silicon)

**What you'll learn:**
- 💡 Why scaling laws matter: predict loss from model size and data size *before* spending millions on training
- 🎯 The Chinchilla scaling law: $L(N, D) = A/N^\alpha + B/D^\beta + E$ — the equation that changed how labs train LLMs
- ⚡ Compute-optimal allocation: given a fixed FLOP budget, how to split between model size and training data
- ⚠️ The emergent abilities debate: are sudden capability jumps real, or a measurement artifact?

💡 **In simple terms**, think of it as building blocks — each concept in this notebook is a building block that connects to the next. For example, understanding the basics here will make everything that follows much easier to grasp.

## Why Scaling Laws Matter

Training a large language model costs millions of dollars. GPT-4 reportedly cost over $100M in compute. Before committing that budget, you want answers to three questions:

1. **How good will the model be?** Given $N$ parameters and $D$ training tokens, what cross-entropy loss should we expect?
2. **How should we allocate compute?** Given a fixed FLOP budget $C$, should we train a bigger model on less data, or a smaller model on more data?
3. **Is it worth scaling further?** At what point do we hit diminishing returns?

💡 **Key insight:** Scaling laws give us *predictive power*. They turn model training from an expensive guess into an engineering optimization problem. Labs like Google DeepMind, OpenAI, and Meta use these equations to plan training runs months in advance.

🎯 **Interview tip:** Understanding scaling laws signals that you think about ML at the systems level — not just "make the model bigger." Interviewers at frontier labs expect you to reason about compute allocation, data requirements, and the relationship between loss and downstream performance.

## 🌍 Power Laws in Everyday Life (Before the Math)

Before we dive into formulas, let's build intuition for what a "power law" is — because it's the foundation of everything in this notebook.

**A power law is a relationship where doubling the input doesn't double the output — it changes by a fixed *ratio*.**

You already know power laws from everyday life:
- **City populations:** A few cities are huge (NYC, Tokyo), but most are small. The 2nd-largest city isn't half the size of the 1st — it follows a power law.
- **Earthquake magnitudes:** A magnitude-6 earthquake is 10× stronger than magnitude-5, not 1 unit stronger.
- **Word frequencies:** The word "the" appears WAY more than "aardvark." The 10th most common word isn't 10× less common than the 1st — it follows a power law.

**Why do we care?** Because model performance follows the same pattern. Making a model 2× bigger doesn't make it 2× better — it improves by a fixed ratio that we can predict with a formula.

### A Concrete Example

Suppose loss follows: L = 100 / N^0.5 (a simple power law)

| Model size (N) | Loss | What happened? |
|----------------|------|----------------|
| 1 | 100.0 | Starting point |
| 4 | 50.0 | 4× bigger → loss halved |
| 16 | 25.0 | 4× bigger again → loss halved again |
| 64 | 12.5 | Same pattern! |

See the pattern? Every time we multiply N by 4, loss divides by 2. That's a power law — **predictable, diminishing returns**.

### The Log-Log Trick

If you plot this on regular axes, you get a curve that's hard to read. But if you plot it on **log-log axes** (where each tick mark represents a 10× increase), the curve becomes a **straight line**. The slope of that line tells you the exponent (how fast things improve).

---

## The Power-Law Phenomenon

Before diving into specific formulas, let's understand the empirical observation that makes scaling laws possible.

When researchers plot cross-entropy loss against model size (or dataset size) on a **log-log scale**, they consistently observe **straight lines**. A straight line on a log-log plot means the relationship follows a **power law**:

$$
L \propto X^{-\alpha} \quad \Longleftrightarrow \quad \log L = -\alpha \log X + \text{const}
$$

where $X$ is either model size $N$ or dataset size $D$, and $\alpha > 0$ is the scaling exponent.

💡 **Why power laws?** This is still an open theoretical question. Some hypotheses:
- Neural networks progressively learn features of decreasing frequency/importance
- The loss landscape has a fractal-like structure where each scale reveals new learnable patterns
- Statistical learning theory predicts power-law generalization bounds in certain regimes

⚠️ **Pitfall:** Power laws only hold within a *scaling regime*. Very small models (< 1M params) and very large models (approaching data exhaustion) can deviate from the trend. Always check that your operating point is within the empirically validated range.

---

## Math Derivation: The Chinchilla Scaling Law

Following the pedagogical pattern: *math derivation → step-by-step code → visualization → comparison*.

### The Core Formula

The Chinchilla scaling law (Hoffmann et al., 2022) models cross-entropy loss as a function of two variables — model parameters $N$ and training tokens $D$:

$$
\boxed{L(N, D) = \frac{A}{N^\alpha} + \frac{B}{D^\beta} + E}
$$

Let's unpack each term:

| Term | Meaning | Interpretation |
|------|---------|----------------|
| $\frac{A}{N^\alpha}$ | **Model capacity term** | Loss due to the model being too small to represent the data distribution. Decreases as a power law in $N$. |
| $\frac{B}{D^\beta}$ | **Data term** | Loss due to insufficient training data. Decreases as a power law in $D$. |
| $E$ | **Irreducible loss** | The entropy of natural language itself. Even a perfect model cannot predict language with zero loss — there is inherent randomness in human text. |

### Calibrated Constants

From the Chinchilla paper (Hoffmann et al., 2022), the fitted constants are:

$$
A \approx 406.4, \quad B \approx 410.7, \quad \alpha \approx 0.34, \quad \beta \approx 0.28, \quad E \approx 1.69
$$

💡 **What do these constants tell us?**
- $\alpha \approx 0.34$: Loss decreases roughly as $N^{-0.34}$. To halve the model-capacity loss, you need to increase $N$ by a factor of $2^{1/0.34} \approx 7.6\times$.
- $\beta \approx 0.28$: Loss decreases roughly as $D^{-0.28}$. Data scaling is slightly less efficient than model scaling ($\beta < \alpha$).
- $E \approx 1.69$: This is the floor — no amount of scaling can push loss below this value. It represents the fundamental unpredictability of natural language (roughly 1.69 nats ≈ 2.44 bits per token).

### Step-by-Step Derivation: Why This Form?

The Chinchilla formula isn't arbitrary — it emerges from a principled decomposition of loss sources. Here's the reasoning:

**Step 1: Decompose the loss into independent sources.**

Assume the total loss can be written as a sum of three independent contributions:

$$
L(N, D) = L_N(N) + L_D(D) + E
$$

where $L_N$ captures the model's representational bottleneck, $L_D$ captures the data bottleneck, and $E$ is the irreducible noise.

**Step 2: Each bottleneck follows a power law.**

Empirically, when we hold $D$ fixed and vary $N$ (or vice versa), the excess loss above $E$ follows a power law:

$$
L_N(N) = \frac{A}{N^\alpha}, \qquad L_D(D) = \frac{B}{D^\beta}
$$

This is the key empirical observation — validated across hundreds of training runs by both Kaplan et al. (2020) and Hoffmann et al. (2022).

**Step 3: Combine into the full formula.**

$$
L(N, D) = \frac{A}{N^\alpha} + \frac{B}{D^\beta} + E
$$

⚡ **Why additive, not multiplicative?** The additive form assumes the model-capacity bottleneck and data bottleneck are approximately independent. If you have a huge model but tiny data, the data term dominates. If you have massive data but a tiny model, the model term dominates. The additive form captures this "weakest link" behavior.

🎯 **Interview tip:** Be ready to explain why the formula has *three* terms, not two. The irreducible loss $E$ is crucial — it prevents the formula from predicting zero loss at infinite scale, which would be physically meaningless.

---

## Compute Budget: The $C \approx 6ND$ Rule

Before optimizing allocation, we need to relate model size and data to compute cost.

### FLOPs Estimation

For a standard transformer with $N$ parameters trained on $D$ tokens, the total floating-point operations are approximately:

$$
\boxed{C \approx 6ND}
$$

**Where does the factor of 6 come from?**

Each training step involves:
1. **Forward pass:** Each parameter participates in ~1 multiply-add per token → $2N$ FLOPs per token
2. **Backward pass:** Computing gradients costs roughly $2\times$ the forward pass → $4N$ FLOPs per token
3. **Total per token:** $2N + 4N = 6N$ FLOPs

Over $D$ tokens:

$$
C = 6N \times D = 6ND \text{ FLOPs}
$$

💡 **Practical scale:** A 7B parameter model trained on 2T tokens requires:
$$
C = 6 \times 7 \times 10^9 \times 2 \times 10^{12} = 8.4 \times 10^{22} \text{ FLOPs}
$$

On an H100 GPU at ~1 PFLOP/s ($10^{15}$ FLOP/s), this takes:
$$
t = \frac{8.4 \times 10^{22}}{10^{15}} = 8.4 \times 10^7 \text{ seconds} \approx 2.7 \text{ years (single GPU)}
$$

With 1000 GPUs in parallel: ~1 day. This is why frontier training requires massive clusters.

⚠️ **Pitfall:** The $6ND$ approximation ignores attention's $O(n^2)$ cost per layer. For very long sequences, the actual compute is higher. But for typical training sequence lengths (2K–8K tokens), the approximation is accurate within ~10%.

---

## Optimal Allocation: Given $C$, Find $N^*$ and $D^*$

This is the central optimization problem of compute-optimal training.

### The Optimization Problem

Given a fixed compute budget $C$, we want to find the model size $N^*$ and token count $D^*$ that minimize loss:

$$
\min_{N, D} \quad L(N, D) = \frac{A}{N^\alpha} + \frac{B}{D^\beta} + E
$$
$$
\text{subject to} \quad 6ND = C
$$

### Solving with Lagrange Multipliers

**Step 1:** Substitute the constraint $D = C / (6N)$ into the loss:

$$
L(N) = \frac{A}{N^\alpha} + \frac{B}{(C/6N)^\beta} + E = \frac{A}{N^\alpha} + \frac{B \cdot (6N)^\beta}{C^\beta} + E
$$

**Step 2:** Take the derivative with respect to $N$ and set it to zero:

$$
\frac{dL}{dN} = -\frac{\alpha A}{N^{\alpha+1}} + \frac{\beta B \cdot 6^\beta \cdot N^{\beta-1}}{C^\beta} = 0
$$

**Step 3:** Solve for $N$:

$$
\frac{\alpha A}{N^{\alpha+1}} = \frac{\beta B \cdot 6^\beta \cdot N^{\beta-1}}{C^\beta}
$$

$$
\alpha A \cdot C^\beta = \beta B \cdot 6^\beta \cdot N^{\alpha + \beta}
$$

$$
N^{\alpha+\beta} = \frac{\alpha A \cdot C^\beta}{\beta B \cdot 6^\beta}
$$

$$
\boxed{N^* = \left(\frac{\alpha A}{\beta B \cdot 6^\beta}\right)^{\frac{1}{\alpha+\beta}} \cdot C^{\frac{\beta}{\alpha+\beta}}}
$$

**Step 4:** Recover $D^*$ from the constraint:

$$
\boxed{D^* = \frac{C}{6 N^*}}
$$

In [ ]:
# 🔍 Before the fancy math: let's FIND the optimal split by trying lots of options
# This builds intuition for WHY the closed-form solution matters

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Our compute budget: enough for a medium-sized training run
C = 1e20  # 10^20 FLOPs

# The Chinchilla constants
A, B, alpha, beta, E = 406.4, 410.7, 0.34, 0.28, 1.69

# Try many different splits of the budget between N and D
# Remember: C = 6 * N * D, so D = C / (6 * N)
N_options = np.logspace(7, 12, 1000)  # Try model sizes from 10M to 1T params

losses = []
for N in N_options:
    D = C / (6.0 * N)  # Given our budget, this is how much data we can afford
    loss = A / (N ** alpha) + B / (D ** beta) + E
    losses.append(loss)

losses = np.array(losses)

# Find the best split
best_idx = np.argmin(losses)
best_N = N_options[best_idx]
best_D = C / (6.0 * best_N)
best_loss = losses[best_idx]

print(f"Budget: {C:.1e} FLOPs")
print(f"Best model size:  N* = {best_N:.2e} parameters")
print(f"Best data size:   D* = {best_D:.2e} tokens")
print(f"Best loss:        L* = {best_loss:.4f}")
print(f"Tokens per param: D*/N* = {best_D/best_N:.1f}")

# Plot the loss landscape — see the sweet spot!
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(N_options, losses, linewidth=2)
ax.axvline(best_N, color='red', linestyle='--', label=f'Optimal N* = {best_N:.2e}')
ax.axhline(best_loss, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Model size N (parameters)')
ax.set_ylabel('Predicted loss')
ax.set_title(f'Finding the Optimal Model Size for Budget C = {C:.1e} FLOPs')
ax.legend()
ax.grid(True, alpha=0.3)
ax.annotate('Too small model\n(wasting compute on data)', 
            xy=(N_options[100], losses[100]), fontsize=9, color='blue', ha='center')
ax.annotate('Too big model\n(not enough data)', 
            xy=(N_options[800], losses[800]), fontsize=9, color='blue', ha='center')
plt.tight_layout()
plt.savefig('figures/18_brute_force_optimal.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n💡 The Lagrange multiplier derivation below finds this sweet spot")
print("   with a formula instead of brute force — faster and exact.")

### Key Insight: How $N^*$ and $D^*$ Scale with Compute

From the formulas above, the optimal allocation scales as:

$$
N^* \propto C^{\frac{\beta}{\alpha+\beta}} \approx C^{0.45}
$$

$$
D^* \propto C^{\frac{\alpha}{\alpha+\beta}} \approx C^{0.55}
$$

Since $\alpha > \beta$ (0.34 > 0.28), **data should scale slightly faster than model size** as compute increases. This is the Chinchilla insight that overturned the Kaplan-era conventional wisdom.

💡 **The Chinchilla ratio:** At the optimum, the ratio of tokens to parameters is approximately:

$$
\frac{D^*}{N^*} \approx 20
$$

This means for every parameter in your model, you should train on roughly 20 tokens. A 7B model should see ~140B tokens. A 70B model should see ~1.4T tokens.

🎯 **Interview tip:** The "20 tokens per parameter" rule is one of the most cited results in modern ML. But note that recent models (LLaMA 3, Gemma) train on *far more* data than Chinchilla suggests — sometimes 100+ tokens per parameter. This works because:
1. Inference cost dominates training cost at scale (smaller model + more data = cheaper to serve)
2. The Chinchilla optimum minimizes *training* compute, not *total lifecycle* cost
3. More data can improve robustness and reduce memorization

---

## Kaplan (2020) vs Chinchilla (2022): The Great Scaling Debate

Two landmark papers proposed different scaling strategies. Understanding their disagreement is essential for interviews.

### Kaplan et al. (2020) — "Scaling Laws for Neural Language Models" (OpenAI)

Kaplan's team proposed a **three separate power laws** formulation:

$$
L(N) = \left(\frac{N_c}{N}\right)^{\alpha_N}, \quad L(D) = \left(\frac{D_c}{D}\right)^{\alpha_D}, \quad L(C) = \left(\frac{C_c}{C}\right)^{\alpha_C}
$$

with $\alpha_N \approx 0.076$, $\alpha_D \approx 0.095$, $\alpha_C \approx 0.050$.

**Kaplan's conclusion:** Model size matters more than data. Given more compute, **scale the model faster than the data**. Specifically, Kaplan recommended:

$$
N \propto C^{0.73}, \quad D \propto C^{0.27}
$$

This led to the "bigger is better" era: GPT-3 (175B params) was trained on only 300B tokens — a ratio of ~1.7 tokens per parameter.

### Hoffmann et al. (2022) — "Chinchilla" (DeepMind)

Chinchilla challenged Kaplan's conclusion by training over 400 models ranging from 70M to 16B parameters, each on 5B to 500B tokens. Their key finding:

$$
N \propto C^{0.45}, \quad D \propto C^{0.55}
$$

**Chinchilla's conclusion:** Data and model size should scale roughly **proportionally**. Kaplan's models were *undertrained* — they used too few tokens relative to model size.

### Why Did Kaplan Get It Wrong?

| Factor | Kaplan (2020) | Chinchilla (2022) |
|--------|--------------|-------------------|
| Training runs | ~100 | ~400 |
| Max model size | 1.5B | 16B |
| Tokens per run | Fixed (short) | Varied systematically |
| LR schedule | Not fully tuned per run | Cosine schedule, tuned per run |
| Key flaw | Didn't vary data enough | Varied both $N$ and $D$ independently |

⚡ **The critical difference:** Kaplan held training tokens roughly constant across model sizes, so their models were increasingly undertrained as $N$ grew. Chinchilla varied both $N$ and $D$ independently, revealing the true optimal ratio.

🎯 **Interview tip:** The Chinchilla result is sometimes summarized as "$N$ should equal $D$" — this is wrong. The correct statement is "$D$ should scale proportionally to $N$," with the constant of proportionality being ~20 tokens per parameter for compute-optimal training.

### Practical Impact: Chinchilla Changed the Industry

The Chinchilla paper had immediate, concrete impact:

| Model | Year | Params | Tokens | Tokens/Param | Chinchilla-optimal? |
|-------|------|--------|--------|--------------|-------------------|
| GPT-3 | 2020 | 175B | 300B | 1.7 | ❌ Severely undertrained |
| Chinchilla | 2022 | 70B | 1.4T | 20 | ✅ Compute-optimal |
| LLaMA 1 | 2023 | 65B | 1.4T | 21.5 | ✅ Near-optimal |
| LLaMA 2 | 2023 | 70B | 2T | 28.6 | ⚡ Over-trained (intentionally) |
| LLaMA 3 | 2024 | 70B | 15T | 214 | ⚡ Heavily over-trained |
| Gemma 4 | 2026 | MoE | — | — | ⚡ Inference-optimized |

💡 **The trend is clear:** Post-Chinchilla models are trained on *more* data than the compute-optimal point suggests. This is because inference cost (which scales with $N$) often dominates the total cost of ownership. A smaller, over-trained model is cheaper to deploy.

⚠️ **Pitfall:** Don't confuse "compute-optimal" with "best." Compute-optimal minimizes training cost for a given loss target. But if you plan to serve the model to millions of users, a smaller model trained longer may be the better economic choice.

---

## Summary of Key Equations

For quick reference, here are the core equations side by side:

| Equation | Formula | Purpose |
|----------|---------|---------|
| **Chinchilla Loss** | $L(N, D) = \frac{A}{N^\alpha} + \frac{B}{D^\beta} + E$ | Predict loss from model size and data |
| **Compute Budget** | $C \approx 6ND$ | Estimate total FLOPs |
| **Optimal $N^*$** | $N^* \propto C^{\beta/(\alpha+\beta)}$ | Compute-optimal model size |
| **Optimal $D^*$** | $D^* \propto C^{\alpha/(\alpha+\beta)}$ | Compute-optimal token count |
| **Chinchilla Ratio** | $D^*/N^* \approx 20$ | Tokens per parameter at optimum |
| **Irreducible Loss** | $E \approx 1.69$ nats | Entropy floor of natural language |

**Calibrated constants:** $A \approx 406.4$, $B \approx 410.7$, $\alpha \approx 0.34$, $\beta \approx 0.28$, $E \approx 1.69$

---

## Setup

### 📦 Library Imports

The next cell loads the libraries we need for this section. Don't worry about memorizing every import — just run the cell and move on. We'll explain each library as we use it.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

mx.random.seed(42)
print(f"MLX version: {mx.__version__}")
print("✅ Ready for Scaling Laws & Compute-Optimal Training!")

---

## 🎯 Implementation: ScalingLawPredictor

Now let's put the math into code. Our `ScalingLawPredictor` (in `utils/scaling.py`) implements three core methods:

1. **`predict_loss(N, D)`** — Computes $L(N, D) = A/N^\alpha + B/D^\beta + E$
2. **`estimate_training_flops(N, D)`** — Computes $C = 6ND$
3. **`compute_optimal_allocation(C)`** — Solves for $N^*$ and $D^*$ given a FLOP budget

💡 All math is pure Python/numpy — no GPU needed for scaling law predictions.

In [ ]:
from utils.scaling import ScalingLawParams, ComputeBudget, ScalingLawPredictor

# Create predictor with Chinchilla defaults
predictor = ScalingLawPredictor()
print(f"Chinchilla params: A={predictor.params.A}, B={predictor.params.B}, "
      f"α={predictor.params.alpha}, β={predictor.params.beta}, E={predictor.params.E}")
print("✅ ScalingLawPredictor ready")

### Predicting Loss for Known Models

Let's use `predict_loss()` to estimate cross-entropy loss for real-world model configurations.

⚡ Notice how loss decreases as either $N$ or $D$ increases — this is the monotonicity property of the scaling law.

In [ ]:
# Predict loss for various (N, D) configurations
configs = [
    ("GPT-2 Small",   124e6,   10e9),
    ("GPT-2 Large",   774e6,   10e9),
    ("GPT-3",         175e9,  300e9),
    ("Chinchilla",     70e9,  1.4e12),
    ("LLaMA-2 70B",    70e9,  2.0e12),
    ("LLaMA-3 70B",    70e9, 15.0e12),
]

print(f"{'Model':<18} {'N':>12} {'D':>12} {'Loss':>8} {'FLOPs':>12}")
print("-" * 66)
for name, N, D in configs:
    loss = predictor.predict_loss(N, D)
    flops = predictor.estimate_training_flops(N, D)
    print(f"{name:<18} {N:>12.2e} {D:>12.2e} {loss:>8.4f} {flops:>12.2e}")

### ⚠️ Handling Common Errors

When working with ML code, errors are normal and expected. Here's a pattern for handling them gracefully — if something goes wrong, you get a helpful message instead of a crash.

In [ ]:
# 💡 Error handling pattern — use this when operations might fail
try:
    # This is where your ML code goes
    import mlx.core as mx
    test = mx.array([1.0, 2.0, 3.0])  # shape: see output
    result = mx.sum(test)  # shape: see output
    mx.eval(result)
    print(f'✅ Success! Result: {result.item()}')
except Exception as e:
    print(f'❌ Error: {e}')
    print('💡 Tip: Check that MLX is installed and your inputs are valid')

### Compute-Optimal Allocation

Given a FLOP budget $C$, `compute_optimal_allocation()` returns the optimal split between model size $N^*$ and training tokens $D^*$.

🎯 **Key check:** The budget conservation property guarantees $|6 \times N^* \times D^* - C| / C \leq 0.10$.

In [ ]:
# Compute optimal allocation across a range of compute budgets
budgets = [1e17, 1e18, 1e19, 1e20, 1e21, 1e22, 1e23, 1e24]

print(f"{'Budget (FLOPs)':>16} {'Optimal N':>14} {'Optimal D':>14} {'D/N Ratio':>10} {'Est. Loss':>10}")
print("-" * 68)
for C in budgets:
    result = predictor.compute_optimal_allocation(C)
    ratio = result.optimal_D / result.optimal_N
    # Verify budget conservation
    actual = 6.0 * result.optimal_N * result.optimal_D
    assert abs(actual - C) / C <= 0.10, "Budget conservation violated!"
    print(f"{C:>16.1e} {result.optimal_N:>14.2e} {result.optimal_D:>14.2e} {ratio:>10.1f} {result.estimated_loss:>10.4f}")

print("\n⚠️ Note: D/N ratio increases with compute — larger budgets favor more data per parameter.")

### 💡 Monotonicity Check: Loss Decreases with Scale

A fundamental property: increasing either $N$ or $D$ (while holding the other fixed) should always reduce loss.

In [ ]:
# Verify monotonicity: loss decreases as N or D increases
D_fixed = 1e12
N_values = np.logspace(7, 11, 20)  # 10M to 100B params
losses_vs_N = [predictor.predict_loss(N, D_fixed) for N in N_values]

N_fixed = 7e9
D_values = np.logspace(9, 13, 20)  # 1B to 10T tokens
losses_vs_D = [predictor.predict_loss(N_fixed, D) for D in D_values]

# Check monotonicity
assert all(losses_vs_N[i] > losses_vs_N[i+1] for i in range(len(losses_vs_N)-1)), "Not monotonic in N!"
assert all(losses_vs_D[i] > losses_vs_D[i+1] for i in range(len(losses_vs_D)-1)), "Not monotonic in D!"
print("✅ Loss is monotonically decreasing in both N and D")
print(f"   Loss range (varying N): {losses_vs_N[-1]:.3f} — {losses_vs_N[0]:.3f}")
print(f"   Loss range (varying D): {losses_vs_D[-1]:.3f} — {losses_vs_D[0]:.3f}")
print(f"   Irreducible loss floor: E = {predictor.params.E}")

---

## 📊 Log-Log Visualizations: Power Laws in Action

💡 **Key insight:** A straight line on a log-log plot is the signature of a power law. These plots let you *see* the scaling exponents α and β — the slopes of the lines tell you how efficiently each resource (parameters or data) reduces loss.

🎯 **Interview tip:** When discussing scaling laws, always mention that the relationships are linear on log-log axes. This is the empirical foundation that makes the entire Chinchilla framework possible.

In [ ]:
from utils.scaling import plot_loss_vs_model_size, plot_loss_vs_data_size

# Loss vs model size (D fixed at 1T tokens)
fig = plot_loss_vs_model_size(predictor, D_fixed=1e12, N_range=np.logspace(7, 11, 30))
plt.show()

The next cell continues the implementation:

**Loss vs data size (N fixed at 7B params)**

In [ ]:
# Loss vs data size (N fixed at 7B params)
fig = plot_loss_vs_data_size(predictor, N_fixed=7e9, D_range=np.logspace(9, 13, 30))
plt.show()

---

## ⚡ Compute Budget Calculator

Given a total FLOP budget, what's the optimal split between model size and training data? The table below answers this for budgets spanning 7 orders of magnitude — from a single-GPU experiment to a frontier training run.

⚠️ **Pitfall:** These are *compute-optimal* allocations that minimize training cost. In practice, many labs intentionally *over-train* smaller models (more tokens per parameter) because inference cost dominates deployment economics.

In [ ]:
from utils.scaling import print_compute_budget_table

# Compute budget calculator across a wide range of FLOP budgets
print_compute_budget_table(predictor, [1e17, 1e18, 1e19, 1e20, 1e21, 1e22, 1e23, 1e24])

---

## 🎯 Kaplan (2020) vs Chinchilla (2022): Visual Comparison

The plots below show the core disagreement between the two scaling law papers. Kaplan recommends scaling model size aggressively ($N \propto C^{0.73}$), while Chinchilla recommends a more balanced split ($N \propto C^{0.45}$).

💡 **What to look for:**
- **Left plot:** Kaplan allocates far more parameters for the same compute budget
- **Right plot:** Chinchilla maintains ~20 tokens/param; Kaplan drops to ~1–2 tokens/param at large scale — severely undertrained models

In [ ]:
from utils.scaling import plot_kaplan_vs_chinchilla

# Compare Kaplan vs Chinchilla optimal allocations
fig = plot_kaplan_vs_chinchilla(np.logspace(17, 24, 30))  # shape: function output
plt.show()

---

## The Emergent Abilities Debate

One of the most heated discussions in modern AI research: do large language models develop **emergent abilities** — capabilities that appear suddenly and unpredictably at a certain scale?

### What Are Emergent Abilities?

An emergent ability is a capability that is **absent in smaller models but present in larger ones**. The key claim is that these abilities don't improve gradually — they jump from near-random performance to above-random performance at a specific model size, with no warning from the scaling curve.

💡 **The intuition:** Imagine testing models of increasing size on 3-digit arithmetic. A 1B model gets 0% correct. A 10B model gets 0% correct. A 50B model gets 0% correct. Then suddenly, a 100B model gets 40% correct. That discontinuous jump is what researchers call "emergence."

### Supporting Evidence: Wei et al. (2022)

Wei et al. ("Emergent Abilities of Large Language Models," 2022, Google) documented dozens of tasks where performance appeared to emerge suddenly:

- **Few-shot arithmetic:** Models below ~100B parameters performed at chance level; above that threshold, accuracy jumped sharply
- **Multi-step reasoning:** Chain-of-thought prompting only helped models above a certain size
- **Word unscrambling, date understanding, logical deduction:** All showed similar phase-transition-like behavior

Their methodology: evaluate models across multiple scales (from millions to hundreds of billions of parameters) on BIG-Bench tasks, using **exact-match accuracy** as the metric.

⚡ **Why this matters for scaling:** If emergent abilities are real, then scaling laws based on cross-entropy loss might miss important capability thresholds. A model could have smoothly decreasing loss but suddenly unlock new capabilities at specific scales — meaning loss alone doesn't tell the full story.

### Opposing Evidence: Schaeffer et al. (2023)

Schaeffer et al. ("Are Emergent Abilities of Large Language Models a Mirage?," 2023, Stanford) challenged the emergence claim with a sharp argument: **the "emergence" is a measurement artifact, not a property of the models.**

Their key insight: the choice of **evaluation metric** creates the illusion of emergence.

- **Binary/exact-match metrics** (e.g., "is the answer exactly correct?") naturally produce sharp transitions. A model that goes from 1% to 5% to 15% accuracy looks like it's "emerging" because all those scores round to "basically zero" on a coarse scale.
- **Continuous metrics** (e.g., token-level edit distance, partial credit scoring) reveal that performance improves **smoothly and predictably** with scale — no sudden jumps.

⚠️ **The core argument:** Emergence is not in the model — it's in the metric. When you measure with a discontinuous ruler, you see discontinuous results. Switch to a continuous ruler, and the "emergence" disappears.

Schaeffer et al. demonstrated this by:
1. Taking the same models and tasks from Wei et al.
2. Replacing exact-match accuracy with continuous metrics (e.g., Brier score, token edit distance)
3. Showing that performance curves became smooth and predictable — no phase transitions

### Where the Debate Stands Today

Both sides have valid points, and the current consensus is nuanced:

| Aspect | Wei et al. (2022) | Schaeffer et al. (2023) |
|--------|-------------------|------------------------|
| **Claim** | Emergent abilities are real | Emergent abilities are a measurement artifact |
| **Metric used** | Exact-match accuracy | Continuous metrics (Brier score, edit distance) |
| **What they showed** | Sharp jumps in task performance at scale | Smooth improvement when measured continuously |
| **Valid point** | Some tasks genuinely require a minimum capability threshold | Metric choice strongly influences whether "emergence" appears |

💡 **The reconciliation:** The debate is fundamentally about **measurement**, not about whether scaling helps. Both sides agree that larger models are more capable. The disagreement is whether the *transition* is sudden (phase transition) or gradual (smooth scaling) — and the answer depends heavily on how you measure.

🎯 **Interview tip:** This is a favorite interview topic at frontier labs. The strong answer is: "Both papers are correct within their framing. Wei et al. showed that binary task metrics exhibit sharp transitions at scale. Schaeffer et al. showed that continuous metrics reveal smooth improvement. The practical takeaway is that scaling laws predict *loss* well, but predicting *task performance* requires careful attention to how you define and measure success."

⚠️ **Pitfall:** Don't dismiss either side. Saying "emergent abilities are definitely real" or "emergent abilities are definitely fake" both miss the nuance. The truth is that the *phenomenon* depends on the *measurement framework*.

---
## ➡️ What's Next?

You've completed this notebook! In **Notebook 19: Reasoning & Test-Time Compute**, we'll explore how models think step by step.

💡 **Before moving on**, make sure you can answer these questions:
- What was the main concept in this notebook?
- Why does it matter for building LLMs?
- Could you explain it to a friend in one sentence?

---

## 📜 History & Alternatives

A chronological tour of scaling laws, compute-optimal training, and the emergent abilities debate — from the first systematic study to the current frontier.

### Timeline

| Year | Paper / Milestone | Team | Key Contribution |
|------|-------------------|------|-----------------|
| **2017** | Transformer ("Attention Is All You Need") | Google Brain | Established the architecture that scaling laws would later study |
| **2020** | **Scaling Laws for Neural Language Models** | OpenAI (Kaplan et al.) | First systematic study of how loss scales with model size, data, and compute. Proposed $N \propto C^{0.73}$ — scale the model aggressively. Led to GPT-3 (175B params, only 300B tokens). |
| **2022** | **Training Compute-Optimal Large Language Models (Chinchilla)** | DeepMind (Hoffmann et al.) | Trained 400+ models to show Kaplan's ratio was wrong. Established $D/N \approx 20$ tokens per parameter. Chinchilla (70B) matched GPT-3 (175B) with 4.5× less compute. Changed how every lab plans training. |
| **2022** | **Emergent Abilities of Large Language Models** | Google (Wei et al.) | Documented tasks where model performance jumps from random to above-random at specific scales. Sparked intense debate about whether scaling produces qualitative leaps in capability. |
| **2023** | **Are Emergent Abilities a Mirage?** | Stanford (Schaeffer et al.) | Argued that emergent abilities are a measurement artifact — switching from binary to continuous metrics makes the "emergence" disappear. Showed smooth, predictable improvement across all scales. |
| **2023** | **LLaMA 1 & 2** | Meta | Applied Chinchilla insights: LLaMA-1 65B trained on 1.4T tokens (~21 tokens/param). LLaMA-2 70B pushed to 2T tokens, intentionally over-training for better inference economics. |
| **2023–2024** | **Beyond Chinchilla: Inference-Optimal Training** | Multiple teams | Recognized that Chinchilla optimizes *training* compute, but *inference* cost often dominates total lifecycle cost. Smaller models trained on far more data (100+ tokens/param) are cheaper to serve. LLaMA-3 70B trained on 15T tokens (~214 tokens/param). |
| **2024** | **Scaling Data-Constrained Language Models** | Muennighoff et al. | Studied what happens when you run out of unique data. Found that repeating data up to ~4 epochs still helps, but with diminishing returns. Proposed data mixing strategies. |
| **2025–2026** | **MoE Scaling & Gemma 4** | Google DeepMind | Mixture of Experts models change the scaling equation: total parameters ≠ active parameters. Gemma 4 (2026) uses MoE to achieve strong performance with fewer active FLOPs per token. |

### Key Takeaways

💡 **The arc of the field:**
1. **Kaplan (2020)** said: "Scale the model, data is secondary" → led to GPT-3 (huge model, relatively little data)
2. **Chinchilla (2022)** said: "Scale model and data equally" → led to Chinchilla, LLaMA-1 (balanced ratio)
3. **Post-Chinchilla (2023+)** said: "Over-train for inference efficiency" → led to LLaMA-3 (small model, massive data)
4. **MoE era (2024+)** said: "Decouple total params from active params" → led to Mixtral, DeepSeek-V3, Gemma 4

⚡ **The trend:** Each generation shifts the optimal strategy. The "right" scaling law depends on what you're optimizing — training compute, inference cost, or total lifecycle cost.

🎯 **Interview tip:** When asked about scaling laws, don't just recite the Chinchilla formula. Show that you understand the *evolution*: Kaplan → Chinchilla → inference-optimal → MoE scaling. Each step refined our understanding of what "optimal" means. The strongest candidates can explain *why* the field moved from one paradigm to the next — it's always driven by economics (training cost vs. serving cost) and practical constraints (data availability, hardware utilization).

⚠️ **What's still open:** We don't have a complete theory of *why* power laws hold. We don't know if current scaling trends will continue to frontier scales (10²⁶+ FLOPs). And the emergent abilities debate remains unresolved — the answer may depend on what kinds of tasks and metrics we care about most.